In [6]:
from IPython.display import display, HTML

display(HTML("""
<div style="
    width:100%;
    min-height:520px;
    box-sizing:border-box;
    padding:65px 75px;
    border-radius:26px;
    background:linear-gradient(135deg,#5F0000 0%,#CC0033 45%,#111827 100%);
    color:white;
    font-family:Arial, Helvetica, sans-serif;
    box-shadow:0 12px 35px rgba(0,0,0,.30);
">

    <div style="
        font-size:20px;
        letter-spacing:3px;
        text-transform:uppercase;
        opacity:.9;
        margin-bottom:35px;
    ">
        AIS Educators Conference 2026
    </div>

    <div style="
        font-size:56px;
        font-weight:800;
        line-height:1.08;
        max-width:950px;
        margin-bottom:30px;
    ">
        Building a RAG Chatbot for AIS Course Support
    </div>

    <div style="
        font-size:25px;
        line-height:1.45;
        max-width:900px;
        opacity:.96;
        margin-bottom:70px;
    ">
        A live Google Colab demonstration of retrieval-augmented generation using course materials.
    </div>

    <div style="
        display:flex;
        gap:18px;
        flex-wrap:wrap;
        margin-bottom:55px;
    ">
        <div style="background:rgba(255,255,255,.15);padding:14px 20px;border-radius:16px;font-size:19px;">
            📄 Course materials
        </div>
        <div style="background:rgba(255,255,255,.15);padding:14px 20px;border-radius:16px;font-size:19px;">
            🔎 Retrieval
        </div>
        <div style="background:rgba(255,255,255,.15);padding:14px 20px;border-radius:16px;font-size:19px;">
            🤖 LLM response
        </div>
        <div style="background:rgba(255,255,255,.15);padding:14px 20px;border-radius:16px;font-size:19px;">
            🎓 AIS education
        </div>
    </div>

    <div style="
        border-top:1px solid rgba(255,255,255,.35);
        padding-top:28px;
        font-size:22px;
        line-height:1.5;
    ">
        <b>Muhammad Talha Afzal</b><br>
        PhD Candidate, Rutgers Business School<br>
        Accounting Information Systems
    </div>

</div>
"""))

In [9]:
from IPython.display import display, HTML

display(HTML("""
<div style="padding:55px 70px;border-radius:24px;background:#ffffff;color:#111827;font-family:Arial;border-left:12px solid #CC0033;box-shadow:0 10px 30px rgba(0,0,0,.18);">
<h1 style="font-size:46px;margin-bottom:25px;">Today's Demonstration</h1>

<ol style="font-size:26px;line-height:1.8;">
<li>Motivation for a course-support chatbot</li>
<li>What retrieval-augmented generation does</li>
<li>How course materials are loaded and chunked</li>
<li>How embeddings and retrieval work</li>
<li>Live chatbot demonstration</li>
<li>Example questions and limitations</li>
</ol>
</div>
"""))

## 1. Install packages

Run this cell, then restart the Colab runtime once before continuing.

In [ ]:
!pip install -q -U \
  langchain==0.3.27 \
  langchain-core==0.3.75 \
  langchain-community==0.3.29 \
  langchain-groq \
  langchain-qdrant \
  qdrant-client \
  fastembed \
  llama-parse \
  gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.0/362.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/

## 2. Imports and API keys

In [ ]:
import os
import getpass
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_groq import ChatGroq
from llama_parse import LlamaParse

/tmp/ipykernel_757/1053417342.py:10: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


In [ ]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
os.environ["LLAMA_CLOUD_API_KEY"] = getpass.getpass("Enter your LlamaCloud API key: ")

Enter your Groq API key: ··········
Enter your LlamaCloud API key: ··········


## 3. Download course files

Replace the folder URL with your own shared Google Drive folder if needed.

In [ ]:
!rm -rf /content/chatbot
!gdown --folder "https://drive.google.com/drive/folders/15_e22ZnN3jH1Kx937CyRGoU1_XlNBVlg" -O /content/chatbot


Retrieving folder contents
Processing file 1S7cjqFeVPUSSo-YSJ38psucK0AVyTaCU en-week1 lecture2.txt
Processing file 1nhLZhtSd5X5xIU1ZmuRnhuqpWsoQoi-Q en-Week1_lecture.txt
Processing file 1vbgj9yml7ztSrR2Z3wdtzFr-KCLVID8L en-Week3_TraditionalData_lec.mp4.txt
Processing file 1Sji-aUq_gXGfkVAdZ6DiTJzm5fMzjJP6 en-Week6_DataPrep2_lec.mp4.txt
Processing file 1wcyXtxvArbdn2Bot8lDKD_sALHeROvld en-week9-alteryx1.mov.txt
Processing file 1jej8ySjmI15XBrb8gdz3Seo6tFLgwD8u en-week9-alteryx2.mov.txt
Processing file 1_5Fy3uV3Cpgs5qhIqPpYhCa8NxFr0Cud en-week9-alteryx3.mov.txt
Processing file 1SnX05odLlzyIVwXW0iIjkA29Dumb2WHz en-week9-Python regression.txt
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1S7cjqFeVPUSSo-YSJ38psucK0AVyTaCU
To: /content/chatbot/en-week1 lecture2.txt
100% 33.1k/33.1k [00:00<00:00, 63.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1nhLZhtSd5X5xIU1ZmuRnhuqpW

In [ ]:
!find /content/chatbot -maxdepth 2 -type f | head -20

/content/chatbot/en-week9-Python regression.txt
/content/chatbot/en-week9-alteryx3.mov.txt
/content/chatbot/en-week1 lecture2.txt
/content/chatbot/en-week9-alteryx1.mov.txt
/content/chatbot/en-Week6_DataPrep2_lec.mp4.txt
/content/chatbot/en-Week1_lecture.txt
/content/chatbot/en-week9-alteryx2.mov.txt
/content/chatbot/en-Week3_TraditionalData_lec.mp4.txt


## 4. Load / parse documents

For `.txt` transcripts, we read them directly. For PDFs, DOCX, PPTX, etc., we use LlamaParse.

In [ ]:
parser = LlamaParse(
    api_key=os.environ["LLAMA_CLOUD_API_KEY"],
    result_type="markdown"
)

raw_docs = []
folder = Path("/content/chatbot")

for filepath in folder.rglob("*"):
    if not filepath.is_file():
        continue

    if filepath.suffix.lower() == ".txt":
        text = filepath.read_text(encoding="utf-8", errors="ignore")
        if text.strip():
            raw_docs.append(Document(page_content=text, metadata={"source": str(filepath)}))
    else:
        parsed = await parser.aload_data(str(filepath))
        for doc in parsed:
            if doc.text.strip():
                raw_docs.append(Document(page_content=doc.text, metadata={"source": str(filepath)}))

print("Raw documents loaded:", len(raw_docs))
print(raw_docs[0].page_content[:500])

Raw documents loaded: 8
Welcome back. In this lecture, I'll
be teaching you how you can use Python
to run analysis which I have a data set that
includes different variables such as
appraisal values. It's related to a
property data set. And this property data
set has several fundamental variables
such as the number of beds the number of
baths square foot occupancy and the more
important one that we're going to use that
i used in my project this is from my own
project that i did and the more important
one is the appraise


## 5. Chunk documents

Chunking makes retrieval more precise. This step is optional if your documents are already small, but it is useful for long lecture transcripts.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(raw_docs)

print("Chunks created:", len(docs))
print(docs[0].page_content[:500])

Chunks created: 43
Welcome back. In this lecture, I'll
be teaching you how you can use Python
to run analysis which I have a data set that
includes different variables such as
appraisal values. It's related to a
property data set. And this property data
set has several fundamental variables
such as the number of beds the number of
baths square foot occupancy and the more
important one that we're going to use that
i used in my project this is from my own
project that i did and the more important
one is the appraise


## 6. Create embeddings and Qdrant vector store

In [ ]:
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

qdrant = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    location=":memory:",
    collection_name="course_documents",
)

retriever = qdrant.as_retriever(search_kwargs={"k": 5})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

## 7. Create the LLM

In [ ]:
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile"
)


## 8. Ask questions over your course documents

This replaces old `RetrievalQA` and avoids the FlashRank compatibility problems.

In [ ]:
def ask_course_bot(question, k=5):
    relevant_docs = retriever.invoke(question)

    context = "\n\n---\n\n".join(
        doc.page_content for doc in relevant_docs[:k]
    )

    prompt = f"""
You are a helpful course assistant. Use only the context below to answer the question.
If the answer is not in the context, say you do not know based on the provided course materials.

Context:
{context}

Question:
{question}

Answer clearly and concisely.
"""

    response = llm.invoke(prompt)
    return response.content, relevant_docs


answer, sources = ask_course_bot(
    "What does the lecture say about fraud?"
)

print(answer)

The lecture mentions that fraud is a topic that can be studied in accounting research, but notes that fraud data is often biased and scarce. It suggests using proxies, such as accruals, to signal manipulation, as direct fraud data may not be available. Additionally, it mentions that fraud investigations take years and not all firms are investigated, which can distort research results.


## 9. Interactive question box

In [ ]:
question = input("Enter your question: ")
answer, sources = ask_course_bot(question)
print(answer)

Enter your question: whats fraud
I do not know based on the provided course materials, as the context does not provide a clear definition of fraud.


## Optional: inspect retrieved source chunks

In [ ]:
results = qdrant.similarity_search_with_score(question, k=5)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"Rank {rank}")
    print(f"Score: {score}")
    print(f"Source: {doc.metadata.get('source')}")
    print(doc.page_content[:300])
    print("="*80)

Rank 1
Score: 0.6732472930217461
Source: /content/chatbot/en-Week3_TraditionalData_lec.mp4.txt
doesn't have to be huge, at least for the
purpose of this course. You can tag the word counts,
for example, and you can associate
word counts with certain outcomes.
For instance, do MD&As, management
discussion and analysis sections that have
a higher would count are they associated
with something y
Rank 2
Score: 0.6443306274777603
Source: /content/chatbot/en-Week1_lecture.txt
specifically. so you will have to think in
terms of audit, what kind of problem are you
solving for auditors, not specifically
accountants, not specifically the
management, not specifically let's say authorities that are looking for
financial crime but specifically
for auditors that will help audito
Rank 3
Score: 0.626579462741546
Source: /content/chatbot/en-Week1_lecture.txt
Auditors have also been able to use analytical
procedures to verify the authenticity
of the figures posted so this is more of an
indirect evidence

In [ ]:
import gradio as gr

def chatbot(question):
    answer, docs = ask_course_bot(question)

    source_text = "\n\n".join(
        [doc.metadata.get("source", "Unknown")
         for doc in docs]
    )

    return answer, source_text

demo = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Retrieved Sources")
    ],
    title="AIS Course RAG Chatbot"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://288730850d7f20de9a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
from IPython.display import display, HTML

display(HTML("""
<div style="
    width:100%;
    min-height:420px;
    box-sizing:border-box;
    padding:55px 60px;
    border-radius:22px;
    background:#FFFFFF;
    color:#111827;
    font-family:Arial, Helvetica, sans-serif;
    border:5px solid #CC0033;
    box-shadow:0 10px 30px rgba(0,0,0,.20);
    text-align:center;
">
    <div style="font-size:54px;font-weight:800;color:#CC0033;margin-top:10px;">
        Thank You!
    </div>

    <div style="font-size:26px;margin-top:20px;color:#374151;">
        Questions and discussion are welcome.
    </div>

    <div style="width:75%;margin:35px auto;border-top:3px solid #111827;"></div>

    <div style="font-size:24px;line-height:1.5;">
        <b>Muhammad Talha Afzal</b><br>
        PhD Candidate, Rutgers Business School<br>
        Accounting Information Systems
    </div>

    <div style="font-size:23px;margin-top:25px;color:#003366;">
        ✉️ ma2028@rutgers.edu
    </div>

    <div style="font-size:18px;margin-top:28px;color:#6B7280;">
        AIS Educators Conference 2026
    </div>
</div>
"""))
